# Time Series Forecasting

# Project description

Sweet Lift Taxi company has collected historical data on taxi orders at airports. To attract more drivers during peak hours, we need to predict the amount of taxi orders for the next hour. Build a model for such a prediction.

The RMSE metric on the test set should not be more than 48.

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.ar_model import AutoReg, ar_select_order
from pmdarima import auto_arima

## Load and View Data

In [ ]:
# Load data
data = pd.read_csv('/datasets/taxi.csv', index_col=[0], parse_dates=[0])

# View data
data.info()
display(data)

Dataset contains 26496 entries, comprising a six month period from March 1, to August 31 2018. Period between observations is 10 min. 

## Preparation

In [ ]:
# Check for missing values
data.isna().sum(axis=0)

# resample to one hour
data = data.resample('1H').sum()
display(data)

In [ ]:
# Split data
train_valid, test = train_test_split(data, shuffle=False, test_size=0.1)
train, valid = train_test_split(train_valid, shuffle=False, test_size=0.111)
print(train.shape)
print(valid.shape)
print(test.shape)

In [ ]:
# Set up RMSE funciton
def RMSE(score):
    rmse = score ** 0.5
    return rmse

## Analysis

In [ ]:
# PLot seasonal data 
decomposition = seasonal_decompose(train)
decomposition.plot()
plt.show()

Decomposed data shows seasonality exists in our data. Because the data is divided into hourly periods it is likely that daily patterns exist. We will plot data for several days to see if this is true. 

In [ ]:
# Plot data for 3 days
N = 3
hour_per_day = 24
hours_in_period = 3 * 24

decomposition.seasonal[0:hours_in_period].plot(title="Daily Taxi Orders")
plt.show()

Plotted data shows repeating daily patterns in taxi orders per hour.

## Training

Training for time series data will be conducted using model classes AutoReg and ARIMA, while using linear regression predictions as baseline.

### Train - Linear Regression

In [ ]:
data_lr = data.copy()

# Create lag funciton
def create_lags(data, max_lag, rolling_mean_size):
    
    #Create comluns
    data['year'] = data.index.year
    data['month'] = data.index.month
    data['day'] = data.index.day
    data['hour'] = data.index.hour
    
    # Specify lag in columns
    for lag in range(1, max_lag + 1):
        data['lag_{}'.format(lag)] = data['num_orders'].shift(lag)
        
    #  Create rolling mean
    data['rolling_mean'] = data['num_orders'].shift().rolling(rolling_mean_size).mean()

# Apply function
create_lags(data_lr, 6, 10)

In [ ]:
# Split lag data
train_valid_lr, test_lr = train_test_split(data_lr, shuffle=False, test_size=0.1)
train_lr, valid_lr = train_test_split(train_valid_lr, shuffle=False, test_size=0.111)

# drop train nans
train_lr = train_lr.dropna()

# Specify features/target
X_train = train_lr.drop('num_orders', axis=1)
y_train = train_lr['num_orders']
X_valid = valid_lr.drop('num_orders', axis=1)
y_valid = valid_lr['num_orders']

In [ ]:
# train linear regression
model_lr = LinearRegression()
model_lr.fit(X_train, y_train)

# Predict MAE
lr_pred = model_lr.predict(X_valid)
lr_mae = mean_absolute_error(y_valid, lr_pred)

# obtain ARIMA RMSE score
print('Valid data MAE:', lr_mae.round(5))
print('Valid data RMSE:', RMSE(lr_mae).round(5))

### Train - AutoReg

In [ ]:
# Find optimal number of lags
for lag in range(30, 60, 10):
    mod = ar_select_order(endog=train, maxlag=lag)
    ar_order = mod.ar_lags

    # train AR model
    ar_model = AutoReg(train, lags=ar_order, seasonal=True)
    ar_model = ar_model.fit()

    # Define start and end points to predict test data
    start_date = len(train)
    end_date = len(train) + len(valid) - 1

    # Predict AR train data
    ar_pred = ar_model.predict(start=start_date, end=end_date, dynamic=False)
    ar_mae = mean_absolute_error(valid, ar_pred)
    print(f'max lag: {lag}, {ar_mae.round(5)}')

In [ ]:
# obtain AutoREg RMSE score
print('Valid data MAE:', ar_mae.round(5))
print('Valid data RMSE:', RMSE(ar_mae).round(5))

In [ ]:
# Visualize results
plt.plot(ar_pred, color='blue', label='pred')
plt.plot(valid, color='red', label='valid')
plt.title("AutoReg - Predictions vs. Validation set")
plt.legend(loc='upper left')
plt.xticks(rotation=70)
plt.show()

### Train - ARIMA

In [ ]:
%%time

# Train ARIMA model
print("running")
arima_model = auto_arima(train, seasonal=True, m=24, scoring='mae')

In [ ]:
# predict ARIMA train data
arima_valid_pred = arima_model.predict(len(valid))
arima_valid_mae = mean_absolute_error(valid, arima_valid_pred)

# obtain ARIMA RMSE score
print('Validation data MAE:', arima_valid_mae.round(5))
print('Validation data RMSE:', RMSE(arima_valid_mae).round(5))

In [ ]:
# Visualize results
plt.plot(arima_valid_pred, color='blue', label='pred')
plt.plot(valid, color='red', label='valid')
plt.title("ARIMA - Predictions vs. Validation set")
plt.legend(loc='upper left')
plt.xticks(rotation=70)
plt.show()

In [ ]:
comp = pd.DataFrame(data={'Model': ['Linear Regression', 'AutoReg', 'ARIMA'], 'MAE': [lr_mae.round(5), ar_mae.round(5), arima_valid_mae.round(5)],
                          'RMSE': [RMSE(lr_mae).round(5), RMSE(ar_mae).round(5), RMSE(arima_valid_mae).round(5)]})
print("\nComparison (Lower is better):")
display(comp)

Root mean squared error between models show Auto Regression model to perfom better. Testing will be then conducted using AutoReg method.

## Testing

In [ ]:
# Find optimal number of lags
mod_test = ar_select_order(endog=train_valid, maxlag=50)
ar_order_test = mod_test.ar_lags

# Test AR model
ar_model_test = AutoReg(train_valid, lags=ar_order_test, seasonal=True)
ar_model_test = ar_model_test.fit()

In [ ]:
# Define start and end points to predict test data
start_value = len(train_valid)
end_value = len(train_valid) + len(test) - 1

# Predict AR test data
ar_pred_test = ar_model_test.predict(start=start_value, end=end_value, dynamic=False)
ar_mae_test = mean_absolute_error(test, ar_pred_test)

In [ ]:
data_test_plot = data.copy()
x = np.zeros(len(data_test_plot))
x[:len(train_valid)] = np.nan
x[len(train_valid):] = ar_pred_test
data_test_plot['predicted_orders'] = x
data_test_plot.rename(columns={'num_orders': 'actual_orders'}, inplace=True)

data_test_plot['2018-06-20 00:00:00':].plot(title='Predicted Taxi Orders')
data_test_plot[len(train_valid):].plot(title='Predicted Taxi Orders')
plt.show()

In [ ]:
# obtain AR RMSE test score
print('Test data MAE:', ar_mae_test.round(5))
print('Test data RMSE:', RMSE(ar_mae_test).round(5))

## Conclussion

Model appears to reach accepcted MAE and RMSE values. This indicates Autoregression model is suited for seasonal taxi order per hour prediction with a MAE of 37.311 and RMSE of 6.108. Further testing is required to refine model.